# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane:** Refresh / Content Opportunity Scoring

I choose this lane because the starter pipeline already shows a clear gap: a simple hand-written rule reaches only ~24% Precision@50, while a learned model roughly triples that. The core decision is “which pages should an editor review first for refresh?”, the unit of analysis is a page, and the output is a ranked review queue with reason codes. This matches the data we already have (trailing-90-day metrics + trend direction) and gives a concrete action someone can take. I can confirm or change the lane until the end of Week 4.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question**  
Which pages should an editor review first for content refresh, given observable search and engagement signals?

**Unit of analysis**  
One page (content item).

**Decision improved**  
Prioritizing the editor’s limited review capacity — which pages get looked at this week.

**Who acts and what they do**  
A content editor or SEO specialist opens the ranked queue, reads the top pages and their reason codes, and decides whether to refresh, rewrite, monitor, or skip.

**Output**  
A ranked review queue: page id + priority score + suggested action + short reason codes + confidence label.

**Cost of a wrong recommendation**  
- False positive (page ranked high but not declining / not worth refreshing): wasted editor hours and possible unnecessary rewrites.  
- False negative (real declining page ranked low): missed traffic loss that could have been recovered earlier.

**Why data / ML helps**  
A single hand rule (e.g. “stale × visible”) captures only one pattern. Real pages show many overlapping signals (age, freshness, position, impressions, CTR, trend). A simple, readable model can combine those signals better than any fixed rule while still remaining explainable. This is ranking / scoring work, measured by Precision@K (how many of the top-K recommendations are actually worth reviewing).

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
import pandas as pd
import os, sys, subprocess

# --- Robust path setup (works in Colab and local) ---
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Clone YOUR repo if it is not already present
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"

    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)
else:
    # Local: walk up until we find the data folder
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV still not found"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")

# 1. How many pages are declining?
declining_rate = (df["trend_direction"].str.lower() == "down").mean()
print(f"\n1. Declining pages (trend_direction == 'down'): {declining_rate:.1%} of the inventory")

# 2. Stale + visible pages
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"2. Stale (≥180 days) AND visible (≥500 impressions): {stale_visible:,} pages")

# 3. Position-tier CTR cliff
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("\n3. Mean CTR by position tier (impressions ≥ 100):")
print(ctr_by_pos.round(3).to_string())

Working directory: /flyrankinternship/flyrankinternship/flyrankinternship
Rows: 30,000 | Columns: 44

1. Declining pages (trend_direction == 'down'): 54.2% of the inventory
2. Stale (≥180 days) AND visible (≥500 impressions): 17 pages

3. Mean CTR by position tier (impressions ≥ 100):
position_tier
page_1      0.355
top_3       0.334
striking    0.256
page_3_5    0.142
deep        0.055


These three numbers show why the lane is worth 7 weeks:
- More than half the inventory is already labeled declining → plenty of positive examples.
- Only a few thousand pages are both stale and still visible → a pure hand rule has limited coverage.
- CTR collapses by position tier → any scoring system must account for position, not just raw volume.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim**
- Observed associations between safe signals (age, freshness, impressions, position, CTR, etc.) and recent trend direction.
- Directional rankings: “pages that look more like historically declining pages.”
- Decision-support: a ranked queue that helps an editor decide where to look first.
- Comparison against a transparent baseline rule on Precision@K under client-holdout validation.

**What I will never claim**
- That I predicted Google’s ranking algorithm.
- That a refresh *caused* recovery (no causal design).
- That any single score is the absolute truth.
- Client names, domains, URLs, or private queries.
- Product flags or health scores as ground truth (they are not in the data and would leak).

All language will stay in the safe set: observed / measured / directional / decision-support.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.